In [ ]:
#Synthetic data create (customers, bank statement, ledger, errors inject)
# ---------- STEP 1: Synthetic data generator ----------
import pandas as pd  # tables (DataFrame) handle panna
import numpy as np   # random numbers generate panna

np.random.seed(42)  # seed fix pannina, ovvoru run-layum same data varum

N_CUST = 3000  # total customers count
N_TXN = 5000   # total bank transactions count

# City list (dashboard-la irukkura same cities)
cities = ['Kolhapur', 'Mumbai', 'Nagpur', 'Nashik', 'Pune', 'Solapur']

# ---------- Customers table ----------
customers = pd.DataFrame({
    'Customer_ID': np.arange(200001, 200001 + N_CUST),  # unique ID 200001-ல இருந்து
    'Branch_ID': np.random.randint(1000, 2000, N_CUST),  # random branch ID
    'City': np.random.choice(cities, N_CUST),  # random city
    'Age_Group': np.random.choice(['Young Adult', 'Adult', 'Senior', 'Unknown'],
                                  N_CUST, p=[0.12, 0.48, 0.36, 0.04]),  # age group split
    'Customer_Type': np.random.choice(['Individual', 'Employee', 'Business', 'Unknown'],
                                      N_CUST, p=[0.30, 0.29, 0.29, 0.12]),  # customer type split
    'Account_Type': np.random.choice(['Savings', 'Current', 'Salary'],
                                     N_CUST, p=[0.5, 0.2, 0.3])  # account type split
})

# ---------- Bank statement (base / correct data) ----------
start = pd.Timestamp('2022-01-01')  # data start date
bank = pd.DataFrame({
    'Date': start + pd.to_timedelta(np.random.randint(0, 1096, N_TXN), unit='D'),  # 2022-2024 (1096 days) random dates
    'Customer_ID': np.random.choice(customers['Customer_ID'], N_TXN),  # random customer
    'Investment_Type': np.random.choice(['Recurring Deposit', 'Fixed Deposit', 'Mutual Fund'], N_TXN),  # investment type
    'Txn_Type': np.random.choice(['Credit', 'Debit'], N_TXN),  # credit or debit
    'Amount': np.round(np.random.uniform(1000, 17500, N_TXN), 2)  # amount 1000 to 17500
})
bank = bank.sort_values('Date').reset_index(drop=True)  # date wise sort pannitu index reset
bank['Bank_Ref'] = ['BNK' + str(i + 1).zfill(5) for i in range(len(bank))]  # BNK00001 style reference

# ---------- Ledger = bank copy, then errors inject ----------
ledger = bank.copy()  # ledger-ah bank data-la irundhu start panrom
ledger['Error_Type'] = 'None'  # first ellame correct nu mark

idx = np.random.permutation(len(ledger))  # row positions-ah shuffle
missing_idx = idx[:150]    # 150 rows ledger-la missing (3%)
amt_idx = idx[150:250]     # 100 rows amount mismatch (2%)
date_idx = idx[250:400]    # 150 rows date shift (3%)
dup_idx = idx[400:450]     # 50 rows duplicate (1%)

# Amount mismatch: amount-ah konjam maathurom (typing mistake maari)
ledger.loc[amt_idx, 'Amount'] = ledger.loc[amt_idx, 'Amount'] + np.random.choice([-500, -100, -50, 50, 100, 500], len(amt_idx))
ledger.loc[amt_idx, 'Error_Type'] = 'Amount Mismatch'  # error label

# Timing difference: ledger date 1-4 days late
ledger.loc[date_idx, 'Date'] = ledger.loc[date_idx, 'Date'] + pd.to_timedelta(np.random.randint(1, 5, len(date_idx)), unit='D')
ledger.loc[date_idx, 'Error_Type'] = 'Timing Difference'  # error label

# Duplicates: same row-ah rendu thadava ledger-la podrom
dups = ledger.loc[dup_idx].copy()  # duplicate rows eduthukurom
dups['Error_Type'] = 'Duplicate in Ledger'  # error label

# Only in Ledger: bank-la illaadha 100 extra entries
extra = pd.DataFrame({
    'Date': start + pd.to_timedelta(np.random.randint(0, 1096, 100), unit='D'),  # random date
    'Customer_ID': np.random.choice(customers['Customer_ID'], 100),  # random customer
    'Investment_Type': np.random.choice(['Recurring Deposit', 'Fixed Deposit', 'Mutual Fund'], 100),  # type
    'Txn_Type': np.random.choice(['Credit', 'Debit'], 100),  # credit/debit
    'Amount': np.round(np.random.uniform(1000, 17500, 100), 2),  # amount
    'Bank_Ref': np.nan,  # bank-la illa
    'Error_Type': 'Only in Ledger'  # error label
})

# Only in Bank (type 1): ledger-la miss aana 150 rows record pannikurom, apparam drop
key_missing = ledger.loc[missing_idx, ['Bank_Ref']].copy()  # missing rows reference
key_missing['Error_Type'] = 'Only in Bank'  # label
ledger = ledger.drop(missing_idx)  # ledger-la irundhu remove

# Ellathaiyum serthu final ledger
ledger = pd.concat([ledger, dups, extra], ignore_index=True)  # original + duplicates + extra
ledger = ledger.sort_values('Date').reset_index(drop=True)  # date wise sort
ledger['Voucher_No'] = ['V' + str(i + 1).zfill(5) for i in range(len(ledger))]  # voucher number

# ---------- Bank charges: bank-la mattum irukkum (ledger-la record panna maranthuduvanga) ----------
charges = pd.DataFrame({
    'Date': start + pd.to_timedelta(np.random.randint(0, 1096, 60), unit='D'),  # random date
    'Customer_ID': np.nan,  # specific customer illa
    'Investment_Type': 'Bank Charges',  # charge type
    'Txn_Type': 'Debit',  # always debit
    'Amount': np.round(np.random.uniform(50, 500, 60), 2),  # small amounts
    'Bank_Ref': ['CHG' + str(i + 1).zfill(4) for i in range(60)]  # CHG0001 style
})
key_charges = charges[['Bank_Ref']].copy()  # charges reference
key_charges['Error_Type'] = 'Only in Bank'  # label
bank = pd.concat([bank, charges], ignore_index=True)  # bank statement-la charges serkurom
bank = bank.sort_values('Date').reset_index(drop=True)  # re-sort

# ---------- Answer key (Step 5 validation-ku) ----------
key_ledger = ledger[['Voucher_No', 'Bank_Ref', 'Error_Type']]  # ledger rows + labels
answer_key = pd.concat([key_ledger, key_missing, key_charges], ignore_index=True)  # ellathaiyum serthu

# ---------- Final files (hidden columns remove) ----------
bank_out = bank[['Bank_Ref', 'Date', 'Customer_ID', 'Investment_Type', 'Txn_Type', 'Amount']]  # bank file columns
ledger_out = ledger[['Voucher_No', 'Date', 'Customer_ID', 'Investment_Type', 'Txn_Type', 'Amount']]  # ledger file columns (Bank_Ref, Error_Type illa)

customers.to_csv('customers.csv', index=False)  # customers save
bank_out.to_csv('bank_statement.csv', index=False)  # bank statement save
ledger_out.to_csv('ledger.csv', index=False)  # ledger save
answer_key.to_csv('answer_key.csv', index=False)  # answer key save

# ---------- Quick check ----------
print('Bank rows   :', len(bank_out))  # bank row count
print('Ledger rows :', len(ledger_out))  # ledger row count
print(answer_key['Error_Type'].value_counts())  # each error type count
print(bank_out.head())  # bank sample
print(ledger_out.head())  # ledger sample

Bank rows   : 5060
Ledger rows : 5000
Error_Type
None                   4600
Only in Bank            210
Timing Difference       150
Only in Ledger          100
Amount Mismatch         100
Duplicate in Ledger      50
Name: count, dtype: int64
   Bank_Ref       Date  Customer_ID    Investment_Type Txn_Type    Amount
0  BNK00001 2022-01-01     202128.0      Fixed Deposit    Debit   5802.94
1  BNK00002 2022-01-01     200823.0      Fixed Deposit   Credit  15531.70
2  BNK00003 2022-01-01     200068.0        Mutual Fund   Credit  10692.60
3  BNK00004 2022-01-01     202825.0  Recurring Deposit    Debit  15941.09
4  BNK00005 2022-01-01     202431.0        Mutual Fund   Credit   6123.86
  Voucher_No       Date  Customer_ID    Investment_Type Txn_Type    Amount
0     V00001 2022-01-01       202128      Fixed Deposit    Debit   5802.94
1     V00002 2022-01-01       200068        Mutual Fund   Credit  10692.60
2     V00003 2022-01-01       202825  Recurring Deposit    Debit  15941.09
3     V00004 

In [ ]:
# ---------- STEP 2: DATA CLEANING ----------
import pandas as pd  # tables handle panna
import numpy as np   # numbers / NaN handle panna

bank = pd.read_csv('bank_statement.csv')       # bank statement load
ledger = pd.read_csv('ledger.csv')             # ledger load

bank['Date'] = pd.to_datetime(bank['Date'])        # bank date-ah proper date format-ku maathurom
ledger['Date'] = pd.to_datetime(ledger['Date'])    # ledger date-ah proper date format-ku maathurom

for df in (bank, ledger):                          # rendu tables-kum same cleaning
    for col in ['Investment_Type', 'Txn_Type']:    # text columns
        df[col] = df[col].astype(str).str.strip().str.title()  # extra space remove + Title Case

bank['Customer_ID'] = bank['Customer_ID'].fillna(-1).astype(int)      # bank charges-ku ID illa, -1 nu fill, float-ah int aakkurom
ledger['Customer_ID'] = ledger['Customer_ID'].fillna(-1).astype(int)  # ledger-kum same

bank['Amt_Paise'] = (bank['Amount'] * 100).round().astype('int64')      # amount-ah paise integer-ah maathurom (float compare problem varaadhu)
ledger['Amt_Paise'] = (ledger['Amount'] * 100).round().astype('int64')  # ledger-kum same

# ---- Data quality checks ----
print('Bank nulls   :', bank.isna().sum().sum())              # bank-la null irukka
print('Ledger nulls :', ledger.isna().sum().sum())            # ledger-la null irukka
print('Dup Bank_Ref :', bank['Bank_Ref'].duplicated().sum())  # bank reference duplicate irukka (0 irukkanum)
print('Dup Voucher  :', ledger['Voucher_No'].duplicated().sum())  # voucher duplicate irukka (0 irukkanum)

# ---------- STEP 3: MATCHING LOGIC ----------
KEYS = ['Customer_ID', 'Investment_Type', 'Txn_Type']  # ellaa passes-um idhu same irukkanum
DATE_TOL = 4      # timing difference max days
AMT_TOL = 500     # amount mismatch max difference (rupees)

b = bank.copy()    # original-ah touch panna vendaam, copy-la work
l = ledger.copy()  # ledger copy

# ---- PASS 1: EXACT MATCH ----
exact_cols = KEYS + ['Date', 'Amt_Paise']              # exact match-ku use panra columns
b['seq'] = b.groupby(exact_cols).cumcount()            # same key rows-ku 0,1,2.. number (one-to-one pairing-ku)
l['seq'] = l.groupby(exact_cols).cumcount()            # ledger-kum same
p1 = b[['Bank_Ref', 'seq'] + exact_cols].merge(
    l[['Voucher_No', 'seq'] + exact_cols],
    on=exact_cols + ['seq'], how='inner')              # key + seq match aanadha pair pannurom
p1 = p1[['Bank_Ref', 'Voucher_No']].copy()             # reference columns mattum
p1['Category'] = 'Matched'                             # category label
p1['Date_Diff'] = 0                                    # date difference 0
p1['Amount_Diff'] = 0.0                                # amount difference 0

# ---- Duplicates in ledger: pass 1-ku apparam meedhi irukkura ledger rows-la, bank-la already irukkura key ----
l_unm = l[~l['Voucher_No'].isin(p1['Voucher_No'])]     # exact match aagaadha ledger rows
chk = l_unm.merge(b[exact_cols].drop_duplicates(), on=exact_cols, how='left', indicator=True)  # bank-la same key irukka check
dup_v = chk.loc[chk['_merge'] == 'both', 'Voucher_No']  # irundha, adhu extra copy = duplicate
dups = pd.DataFrame({'Bank_Ref': np.nan, 'Voucher_No': dup_v.values})  # duplicate rows table
dups['Category'] = 'Duplicate in Ledger'               # category label
dups['Date_Diff'] = np.nan                             # n/a
dups['Amount_Diff'] = np.nan                           # n/a

# ---- Remaining pools ----
b_rem = b[~b['Bank_Ref'].isin(p1['Bank_Ref'])]                                          # bank-la innum match aagaadhavai
l_rem = l[~l['Voucher_No'].isin(p1['Voucher_No']) & ~l['Voucher_No'].isin(dup_v)]       # ledger-la match aagaadhavai (duplicates thavira)

# ---- PASS 2: TIMING DIFFERENCE (amount same, date +-4 days) ----
c = b_rem[['Bank_Ref', 'Date'] + KEYS + ['Amt_Paise']].merge(
    l_rem[['Voucher_No', 'Date'] + KEYS + ['Amt_Paise']],
    on=KEYS + ['Amt_Paise'], suffixes=('_bank', '_ledger'))   # same customer/type/amount candidates
c['Date_Diff'] = (c['Date_ledger'] - c['Date_bank']).dt.days  # ledger date - bank date (days)
c = c[c['Date_Diff'].abs() <= DATE_TOL].copy()                # 4 days-kulla irukkaradhu mattum
c['abs_d'] = c['Date_Diff'].abs()                             # nearest date-ku priority kudukka
c = c.sort_values('abs_d').drop_duplicates('Bank_Ref').drop_duplicates('Voucher_No')  # one-to-one pairing
p2 = c[['Bank_Ref', 'Voucher_No', 'Date_Diff']].copy()        # result columns
p2['Category'] = 'Timing Difference'                          # category label
p2['Amount_Diff'] = 0.0                                       # amount same

b_rem = b_rem[~b_rem['Bank_Ref'].isin(p2['Bank_Ref'])]        # timing matched rows remove
l_rem = l_rem[~l_rem['Voucher_No'].isin(p2['Voucher_No'])]    # timing matched rows remove

# ---- PASS 3: AMOUNT MISMATCH (date same, amount differs <= 500) ----
c = b_rem[['Bank_Ref', 'Date'] + KEYS + ['Amt_Paise']].merge(
    l_rem[['Voucher_No', 'Date'] + KEYS + ['Amt_Paise']],
    on=KEYS + ['Date'], suffixes=('_bank', '_ledger'))        # same customer/type/date candidates
c['Amount_Diff'] = (c['Amt_Paise_ledger'] - c['Amt_Paise_bank']) / 100  # ledger - bank amount (rupees)
c = c[c['Amount_Diff'].abs() <= AMT_TOL].copy()               # 500-kulla irukkaradhu mattum
c['abs_a'] = c['Amount_Diff'].abs()                           # smallest difference-ku priority
c = c.sort_values('abs_a').drop_duplicates('Bank_Ref').drop_duplicates('Voucher_No')  # one-to-one pairing
p3 = c[['Bank_Ref', 'Voucher_No', 'Amount_Diff']].copy()      # result columns
p3['Category'] = 'Amount Mismatch'                            # category label
p3['Date_Diff'] = 0                                           # date same

b_rem = b_rem[~b_rem['Bank_Ref'].isin(p3['Bank_Ref'])]        # amount matched rows remove
l_rem = l_rem[~l_rem['Voucher_No'].isin(p3['Voucher_No'])]    # amount matched rows remove

# ---- LEFTOVERS ----
only_bank = pd.DataFrame({'Bank_Ref': b_rem['Bank_Ref'].values, 'Voucher_No': np.nan})   # bank-la mattum irukkaradhu
only_bank['Category'] = 'Only in Bank'                                                   # label
only_ledger = pd.DataFrame({'Bank_Ref': np.nan, 'Voucher_No': l_rem['Voucher_No'].values})  # ledger-la mattum irukkaradhu
only_ledger['Category'] = 'Only in Ledger'                                               # label

# ---- Ellathaiyum serthu final result ----
result = pd.concat([p1, p2, p3, dups, only_bank, only_ledger], ignore_index=True)  # all categories oru table-la

bank_det = bank[['Bank_Ref', 'Date', 'Amount', 'Customer_ID', 'Investment_Type', 'Txn_Type']].rename(
    columns={'Date': 'Bank_Date', 'Amount': 'Bank_Amount', 'Customer_ID': 'B_Cust',
             'Investment_Type': 'B_Inv', 'Txn_Type': 'B_Txn'})                    # bank details rename
led_det = ledger[['Voucher_No', 'Date', 'Amount', 'Customer_ID', 'Investment_Type', 'Txn_Type']].rename(
    columns={'Date': 'Ledger_Date', 'Amount': 'Ledger_Amount', 'Customer_ID': 'L_Cust',
             'Investment_Type': 'L_Inv', 'Txn_Type': 'L_Txn'})                    # ledger details rename

result = result.merge(bank_det, on='Bank_Ref', how='left')       # bank details attach
result = result.merge(led_det, on='Voucher_No', how='left')      # ledger details attach
result['Customer_ID'] = result['B_Cust'].fillna(result['L_Cust']).astype(int)   # bank-la illana ledger customer
result['Investment_Type'] = result['B_Inv'].fillna(result['L_Inv'])             # same logic
result['Txn_Type'] = result['B_Txn'].fillna(result['L_Txn'])                    # same logic
result = result.drop(columns=['B_Cust', 'L_Cust', 'B_Inv', 'L_Inv', 'B_Txn', 'L_Txn'])  # helper columns remove

# ---- Sanity checks ----
print('Bank rows accounted   :', result['Bank_Ref'].notna().sum(), '/', len(bank))        # 5060 irukkanum
print('Ledger rows accounted :', result['Voucher_No'].notna().sum(), '/', len(ledger))    # 5000 irukkanum
print(result['Category'].value_counts())                                                  # category wise count

result.to_csv('recon_result.csv', index=False)  # next steps-ku save
print(result.head(10))                          # sample rows


Bank nulls   : 0
Ledger nulls : 0
Dup Bank_Ref : 0
Dup Voucher  : 0
Bank rows accounted   : 5060 / 5060
Ledger rows accounted : 5000 / 5000
Category
Matched                4600
Only in Bank            210
Timing Difference       150
Amount Mismatch         100
Only in Ledger          100
Duplicate in Ledger      50
Name: count, dtype: int64
   Bank_Ref Voucher_No Category  Date_Diff  Amount_Diff  Bank_Date  \
0  BNK00001     V00001  Matched        0.0          0.0 2022-01-01   
1  BNK00003     V00002  Matched        0.0          0.0 2022-01-01   
2  BNK00004     V00003  Matched        0.0          0.0 2022-01-01   
3  BNK00006     V00004  Matched        0.0          0.0 2022-01-01   
4  BNK00007     V00005  Matched        0.0          0.0 2022-01-01   
5  BNK00008     V00007  Matched        0.0          0.0 2022-01-02   
6  BNK00009     V00008  Matched        0.0          0.0 2022-01-02   
7  BNK00010     V00010  Matched        0.0          0.0 2022-01-03   
8  BNK00011     V00011  Mat

In [ ]:
# xlsxwriter library install panrom (Excel formatting-ku venum)
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.6 MB/s eta 0:00:00


In [ ]:
# ---------- STEP 4: RECONCILIATION STATEMENT + EXCEL REPORT ----------
import pandas as pd  # tables handle panna
import numpy as np   # numbers handle panna

# ---- Files reload (safe-ah irukka) ----
result = pd.read_csv('recon_result.csv', parse_dates=['Bank_Date', 'Ledger_Date'])  # Step 3 result load
bank = pd.read_csv('bank_statement.csv')      # bank statement load
ledger = pd.read_csv('ledger.csv')            # ledger load

for df in (bank, ledger):                     # rendu tables-kum cleaning
    df['Txn_Type'] = df['Txn_Type'].astype(str).str.strip().str.title()  # Credit / Debit standardize
result['Txn_Type'] = result['Txn_Type'].astype(str).str.strip().str.title()  # result-kum same

# ---- Signed amounts: Credit = +, Debit = - (balance calculate panna) ----
bank['Signed'] = np.where(bank['Txn_Type'] == 'Credit', bank['Amount'], -bank['Amount'])        # bank signed amount
ledger['Signed'] = np.where(ledger['Txn_Type'] == 'Credit', ledger['Amount'], -ledger['Amount'])  # ledger signed amount
result['Sign'] = np.where(result['Txn_Type'] == 'Credit', 1, -1)                                # +1 / -1 sign column
result['Txn_Amount'] = result['Bank_Amount'].fillna(result['Ledger_Amount'])   # bank amount, illana ledger amount
result['Event_Date'] = result['Bank_Date'].fillna(result['Ledger_Date'])       # bank date, illana ledger date
result['Month'] = result['Event_Date'].dt.to_period('M').astype(str)           # month label (2022-01 style)

# ---- Reconciliation statement calculation ----
cat = result['Category']  # category column shortcut

def signed_sum(mask, col):                                       # helper: signed total for given rows
    return (result.loc[mask, 'Sign'] * result.loc[mask, col]).sum()  # sign x amount, then sum

bank_bal = bank['Signed'].sum()        # balance as per bank statement
ledger_bal = ledger['Signed'].sum()    # balance as per ledger (actual)

adj_only_bank = -signed_sum(cat == 'Only in Bank', 'Bank_Amount')          # bank-la irukku, ledger-la illa -> ledger-ku vara minus
adj_only_ledger = signed_sum(cat == 'Only in Ledger', 'Ledger_Amount')     # ledger-la mattum irukku -> plus
adj_dup = signed_sum(cat == 'Duplicate in Ledger', 'Ledger_Amount')        # ledger-la extra copy -> plus
m = cat == 'Amount Mismatch'                                               # amount mismatch rows mask
adj_amt = (result.loc[m, 'Sign'] * result.loc[m, 'Amount_Diff']).sum()     # (ledger - bank) difference, signed
adj_timing = 0.0                                                           # same amount, date mattum differ -> total balance-la effect illa

calc_ledger = bank_bal + adj_only_bank + adj_only_ledger + adj_dup + adj_amt + adj_timing  # bank + adjustments = expected ledger
diff = round(calc_ledger - ledger_bal, 2)   # expected vs actual (0.00 irukkanum)

stmt = pd.DataFrame({                       # statement table
    'Particulars': [
        'Balance as per Bank Statement',
        'Adj: Only in Bank (bank charges etc., not booked in ledger)',
        'Adj: Only in Ledger (not found in bank)',
        'Adj: Duplicate entries in Ledger',
        'Adj: Amount mismatches (ledger vs bank)',
        'Adj: Timing differences (date only, no balance impact)',
        'Balance as per Ledger (calculated)',
        'Balance as per Ledger (actual)',
        'Unexplained Difference'],
    'Amount (INR)': [bank_bal, adj_only_bank, adj_only_ledger, adj_dup, adj_amt, adj_timing,
                     calc_ledger, ledger_bal, diff],   # amounts in same order
    'Remarks': [
        'Credits minus Debits',
        'Ledger-la entry pottu correct pannanum',
        'Verify with bank / reverse entry',
        'Reverse the duplicate voucher',
        'Correct the ledger amount',
        'Cut-off date-la mattum matter aagum',
        'Bank balance + adjustments',
        'Credits minus Debits',
        'Should be 0.00']}).round(2)             # remarks + rounding

# ---- Exceptions table ----
REPORT_DATE = pd.Timestamp('2024-12-31')   # reporting date (data end date)
actions = {                                 # category -> suggested action
    'Timing Difference': 'Wait for clearing; monitor next cycle',
    'Amount Mismatch': 'Check source document, correct ledger amount',
    'Duplicate in Ledger': 'Reverse duplicate voucher',
    'Only in Bank': 'Book entry in ledger (charges / missed entry)',
    'Only in Ledger': 'Verify with bank; reverse if invalid'}

exc = result[cat != 'Matched'].copy()                                     # matched illaadha ellam exceptions
exc['Age_Days'] = (REPORT_DATE - exc['Event_Date']).dt.days.clip(lower=0)  # report date-la irundhu evlo naal aachu
labels = ['0-30', '31-90', '91-180', '181-365', '365+']                    # aging buckets
exc['Age_Bucket'] = pd.cut(exc['Age_Days'], bins=[-1, 30, 90, 180, 365, np.inf], labels=labels).astype(str)  # bucket assign
exc['Suggested_Action'] = exc['Category'].map(actions)                     # action text
exc['Customer_ID'] = exc['Customer_ID'].replace(-1, 'N/A')                 # bank charges-ku customer illa
exc = exc[['Category', 'Bank_Ref', 'Voucher_No', 'Event_Date', 'Customer_ID', 'Investment_Type',
           'Txn_Type', 'Bank_Amount', 'Ledger_Amount', 'Amount_Diff', 'Date_Diff',
           'Age_Days', 'Age_Bucket', 'Suggested_Action']].sort_values(['Category', 'Event_Date'])  # column order + sort

# ---- Summary tables ----
n_bank, n_ledger = len(bank), len(ledger)                       # total counts
n_exact = int((cat == 'Matched').sum())                         # exact matched count
n_recon = int(cat.isin(['Matched', 'Timing Difference', 'Amount Mismatch']).sum())  # bank rows with ledger pair
kpi = pd.DataFrame({                                            # KPI table
    'KPI': ['Total Bank Transactions', 'Total Ledger Entries', 'Exact Matched',
            'Exact Match %', 'Reconciled % (incl. timing & amount mismatch)',
            'Total Exceptions', 'Balance as per Bank', 'Balance as per Ledger', 'Unexplained Difference'],
    'Value': [f'{n_bank:,}', f'{n_ledger:,}', f'{n_exact:,}',
              f'{n_exact / n_bank:.2%}', f'{n_recon / n_bank:.2%}',
              f'{len(exc):,}', f'{bank_bal:,.2f}', f'{ledger_bal:,.2f}', f'{diff:,.2f}']})  # formatted values

cat_sum = (result.groupby('Category')                           # category wise summary
           .agg(Count=('Category', 'size'), Total_Amount=('Txn_Amount', 'sum'))  # count + total amount
           .reset_index())                                      # category normal column-ah maathurom
cat_sum['Pct_of_Rows'] = (cat_sum['Count'] / cat_sum['Count'].sum() * 100).round(2)  # % share
cat_sum['Total_Amount'] = cat_sum['Total_Amount'].round(2)      # amount rounding

aging = (pd.crosstab(exc['Category'], exc['Age_Bucket'])        # category x bucket counts
         .reindex(columns=labels, fill_value=0).reset_index())  # bucket order fix

monthly = pd.crosstab(result['Month'], result['Category']).reset_index()  # month x category counts

# ---- Excel writing ----
out = 'Bank_Ledger_Reconciliation_Report.xlsx'                  # output file name
writer = pd.ExcelWriter(out, engine='xlsxwriter', datetime_format='yyyy-mm-dd')  # date format clean-ah
wb = writer.book                                                # workbook object
hdr = wb.add_format({'bold': True, 'bg_color': '#1F4E78', 'font_color': 'white', 'border': 1})  # header style
money = wb.add_format({'num_format': '#,##0.00'})               # amount format

def write_df(df, sheet, startrow=0, fit=True):                  # helper: sheet-ku table write + format
    df.to_excel(writer, sheet_name=sheet, index=False, startrow=startrow)  # data write
    ws = writer.sheets[sheet]                                   # worksheet object
    for i, col in enumerate(df.columns):                        # ovvoru column-kum
        ws.write(startrow, i, col, hdr)                         # header style apply
        if fit:                                                 # width + number format
            w = max(len(str(col)), df[col].astype(str).str.len().max()) + 2   # column width
            is_money = any(k in col for k in ['Amount', 'Balance'])           # amount column-aa
            ws.set_column(i, i, min(w, 55), money if is_money else None)      # width + format
    return ws                                                   # worksheet return

ws = write_df(kpi, 'Summary', 0, fit=False)                     # KPI table
write_df(cat_sum, 'Summary', len(kpi) + 3, fit=False)           # category table keezha
ws.set_column(0, 0, 45)                                         # summary col A width
ws.set_column(1, 4, 20)                                         # summary other columns width

write_df(stmt, 'Recon_Statement')                               # reconciliation statement sheet
for name, df_, in [('Exceptions', exc), ('Aging', aging), ('Monthly_Trend', monthly), ('Full_Result', result)]:
    w_ = write_df(df_, name)                                    # data sheets write
    w_.freeze_panes(1, 0)                                       # header freeze
    w_.autofilter(0, 0, len(df_), len(df_.columns) - 1)         # filter buttons

writer.close()                                                  # file save
result.to_csv('recon_final.csv', index=False)                   # Power BI-ku CSV copy

# ---- Output check ----
print(stmt.to_string(index=False))                              # statement print
print('\nUnexplained difference:', diff)                         # 0.0 irukkanum
print(kpi.to_string(index=False))                               # KPI print
print('Saved:', out)                                            # file name

from google.colab import files                                  # Colab download helper
files.download(out)                                             # Excel download

                                                Particulars  Amount (INR)                                Remarks
                              Balance as per Bank Statement     423860.95                   Credits minus Debits
Adj: Only in Bank (bank charges etc., not booked in ledger)     -29156.59 Ledger-la entry pottu correct pannanum
                    Adj: Only in Ledger (not found in bank)     277970.35       Verify with bank / reverse entry
                           Adj: Duplicate entries in Ledger     -44157.88          Reverse the duplicate voucher
                    Adj: Amount mismatches (ledger vs bank)      -2000.00              Correct the ledger amount
     Adj: Timing differences (date only, no balance impact)          0.00    Cut-off date-la mattum matter aagum
                         Balance as per Ledger (calculated)     626516.83             Bank balance + adjustments
                             Balance as per Ledger (actual)     626516.83                   Cred

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ---------- STEP 5: VALIDATION vs ANSWER KEY ----------
import pandas as pd  # tables handle panna

result = pd.read_csv('recon_result.csv')   # Step 3 matching result load
ak = pd.read_csv('answer_key.csv')         # Step 1 answer key load

ak['Error_Type'] = ak['Error_Type'].fillna('None')  # pandas "None" text-ah NaN-ah padikkum, so thirumba "None" nu fill

# ---- Answer key-ah rendu partha pirikkurom ----
truth_ledger = ak[ak['Voucher_No'].notna()][['Voucher_No', 'Error_Type']]  # ledger-la irukkura rows (voucher unique)
truth_bank = ak[ak['Voucher_No'].isna()][['Bank_Ref', 'Error_Type']]       # bank-la mattum irukkura rows (Only in Bank)

# ---- Result-oda join pannurom ----
pred_l = result[result['Voucher_No'].notna()].merge(truth_ledger, on='Voucher_No', how='left')  # ledger rows + true label
pred_b = result[result['Voucher_No'].isna()].merge(truth_bank, on='Bank_Ref', how='left')       # bank-only rows + true label
check = pd.concat([pred_l, pred_b], ignore_index=True)   # rendaiyum serthu

check['True_Category'] = check['Error_Type'].replace({'None': 'Matched'})  # answer key "None" = correctly matched
check['Correct'] = check['Category'] == check['True_Category']             # namma label = true label-aa?

# ---- Results ----
print('Rows validated :', len(check))                              # 5210 irukkanum
print('Unlabeled rows :', check['True_Category'].isna().sum())     # 0 irukkanum (key-la illaadhavai)
print('Correct        :', int(check['Correct'].sum()))             # correct count
print('Accuracy       : {:.2%}'.format(check['Correct'].mean()))   # overall accuracy

print('\nConfusion matrix (rows = true, columns = predicted):')    # heading
print(pd.crosstab(check['True_Category'], check['Category']))      # entha category-ku enna predict aachu

print('\nPer-category accuracy:')                                  # heading
print(check.groupby('True_Category')['Correct'].mean().map('{:.2%}'.format))  # category wise accuracy

wrong = check[~check['Correct']]                                   # thappaa classify aanavai
print('\nMisclassified rows:', len(wrong))                        # count
if len(wrong) > 0:                                                 # irundha mattum
    print(wrong[['Bank_Ref', 'Voucher_No', 'Category', 'True_Category']].head(20))  # first 20 kaatu

check.to_csv('validation_result.csv', index=False)                 # validation file save

Rows validated : 5210
Unlabeled rows : 0
Correct        : 5164
Accuracy       : 99.12%

Confusion matrix (rows = true, columns = predicted):
Category             Amount Mismatch  Duplicate in Ledger  Matched  \
True_Category                                                        
Amount Mismatch                  100                    0        0   
Duplicate in Ledger                0                   27       23   
Matched                            0                   23     4577   
Only in Bank                       0                    0        0   
Only in Ledger                     0                    0        0   
Timing Difference                  0                    0        0   

Category             Only in Bank  Only in Ledger  Timing Difference  
True_Category                                                         
Amount Mismatch                 0               0                  0  
Duplicate in Ledger             0               0                  0  
Matched       

In [ ]:
# ---------- STEP 5b: GROUP-LEVEL VALIDATION (identical twin fix) ----------
import pandas as pd  # tables handle panna

check = pd.read_csv('validation_result.csv')   # Step 5 validation file load
ledger = pd.read_csv('ledger.csv')             # ledger load (twin key create panna)

ledger['Customer_ID'] = ledger['Customer_ID'].fillna(-1).astype(int)   # Customer_ID int aakkurom
ledger['Investment_Type'] = ledger['Investment_Type'].astype(str).str.strip().str.title()  # text clean
ledger['Txn_Type'] = ledger['Txn_Type'].astype(str).str.strip().str.title()                # text clean

ledger['Twin_Key'] = (ledger['Customer_ID'].astype(str) + '|' + ledger['Investment_Type'] + '|' +
                      ledger['Txn_Type'] + '|' + ledger['Date'].astype(str) + '|' +
                      ledger['Amount'].map('{:.2f}'.format))   # identical rows-ku same key

check = check.merge(ledger[['Voucher_No', 'Twin_Key']], on='Voucher_No', how='left')  # twin key attach

# ---- True duplicate irukkura groups mattum edukkurom ----
dup_groups = check.loc[check['True_Category'] == 'Duplicate in Ledger', 'Twin_Key'].unique()  # duplicate irukkura keys
in_dup = check['Twin_Key'].isin(dup_groups)    # andha groups-oda rows

# ---- Group OK: exactly 1 'Duplicate in Ledger', baaki ellam 'Matched' ----
def group_ok(s):                                                   # helper function per group
    return (s == 'Duplicate in Ledger').sum() == 1 and (s == 'Matched').sum() == len(s) - 1  # rule check

g = check[in_dup].groupby('Twin_Key')['Category'].apply(group_ok)  # ovvoru group-kum True / False
check['Correct_Adj'] = check['Correct']                            # first original result copy
check.loc[in_dup, 'Correct_Adj'] = check.loc[in_dup, 'Twin_Key'].map(g).astype(bool)  # group rows-ku group result

# ---- Results ----
print('Duplicate groups checked      :', len(dup_groups))                       # 50 irukkanum
print('Groups correctly resolved     :', int(g.sum()), '/', len(g))             # 50 / 50 irukkanum
print('Predicted Duplicates total    :', int((check['Category'] == 'Duplicate in Ledger').sum()))  # 50 irukkanum
print('Strict accuracy (row label)   : {:.2%}'.format(check['Correct'].mean()))      # 99.12%
print('Adjusted accuracy (group lvl) : {:.2%}'.format(check['Correct_Adj'].mean()))  # 100%

check.to_csv('validation_result_adj.csv', index=False)   # final validation file save

Duplicate groups checked      : 50
Groups correctly resolved     : 50 / 50
Predicted Duplicates total    : 50
Strict accuracy (row label)   : 99.12%
Adjusted accuracy (group lvl) : 100.00%


In [ ]:
# ---------- STEP 6A: POWER BI-KU DATA PREP ----------
import pandas as pd  # tables handle panna
import numpy as np   # numbers / NaN handle panna

res = pd.read_csv('recon_final.csv', parse_dates=['Bank_Date', 'Ledger_Date', 'Event_Date'])  # Step 4 final result load
cust = pd.read_csv('customers.csv')   # Step 1 customers load

REPORT_DATE = pd.Timestamp('2024-12-31')   # reporting date (data end date)

res['Is_Exception'] = (res['Category'] != 'Matched').astype(int)   # matched illaadhavai = 1, matched = 0
res['Year'] = res['Event_Date'].dt.year                             # year column (slicer-ku)

# ---- Aging (exceptions-ku mattum) ----
res['Age_Days'] = np.where(res['Is_Exception'] == 1,                                  # exception-na
                           (REPORT_DATE - res['Event_Date']).dt.days.clip(lower=0),   # report date-la irundhu evlo naal
                           np.nan)                                                    # matched-ku blank
labels = ['0-30', '31-90', '91-180', '181-365', '365+']                               # bucket names
bucket = pd.cut(res['Age_Days'], bins=[-1, 30, 90, 180, 365, np.inf], labels=labels).astype(object)  # bucket assign
res['Age_Bucket'] = bucket.where(bucket.notna(), 'N/A')                               # matched rows-ku N/A
res['Age_Bucket_Order'] = res['Age_Bucket'].map({'N/A': 0, '0-30': 1, '31-90': 2, '91-180': 3, '181-365': 4, '365+': 5})  # sort order

# ---- Category sort order (chart-la oru fixed order-la varum) ----
order = {'Matched': 1, 'Timing Difference': 2, 'Amount Mismatch': 3,
         'Duplicate in Ledger': 4, 'Only in Bank': 5, 'Only in Ledger': 6}   # order mapping
res['Category_Order'] = res['Category'].map(order)                          # order column

# ---- Suggested action ----
actions = {'Timing Difference': 'Wait for clearing; monitor next cycle',
           'Amount Mismatch': 'Check source document, correct ledger amount',
           'Duplicate in Ledger': 'Reverse duplicate voucher',
           'Only in Bank': 'Book entry in ledger (charges / missed entry)',
           'Only in Ledger': 'Verify with bank; reverse if invalid'}          # category -> action
res['Suggested_Action'] = res['Category'].map(actions).fillna('No action needed')  # matched-ku no action

# ---- Customers dimension: -1 (bank charges) row serkrom ----
na_row = pd.DataFrame([{'Customer_ID': -1, 'Branch_ID': 0, 'City': 'Bank Charges / N/A',
                        'Age_Group': 'N/A', 'Customer_Type': 'N/A', 'Account_Type': 'N/A'}])  # N/A customer row
cust_dim = pd.concat([cust, na_row], ignore_index=True)   # customers + N/A row

res.to_csv('recon_dashboard.csv', index=False)    # fact table save
cust_dim.to_csv('customers_dim.csv', index=False) # dimension table save

print(res.shape, cust_dim.shape)                  # rows x columns check
print(res['Age_Bucket'].value_counts())           # bucket counts check

from google.colab import files                    # Colab download helper
files.download('recon_dashboard.csv')             # fact table download
files.download('customers_dim.csv')               # dimension download

(5210, 23) (3001, 6)
Age_Bucket
N/A        4600
365+        414
181-365     106
91-180       46
31-90        29
0-30         15
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>